# VietLangAI – 09B | Kaggle GPU: Đánh giá FULL Validation của ByT5 + LoRA

**Không train lại. Không đọc tập test.** Notebook này chạy trên Kaggle GPU P100 hoặc T4, tải `google/byt5-small` từ Hugging Face và ghép **adapter LoRA** từ dataset riêng tư bạn đính kèm.

### Chuẩn bị một lần

1. Chạy notebook **09A – Đóng gói dữ liệu trên Colab CPU**, tải `VietLangAI_Kaggle_Eval_Inputs.zip` về máy.
2. Vào Kaggle → **Datasets → New Dataset** → tải ZIP lên, để **Private**. Kaggle thường giải nén ZIP thành các file dữ liệu.
3. Vào [Kaggle Code](https://www.kaggle.com/code) → **New Notebook** → **File → Import Notebook** để nhập file này.
4. Trong notebook → **Settings / Session options** → **Accelerator: GPU (P100 hoặc T4)**; **Internet: On** để tải base ByT5. Nếu không có GPU thì xem quota/xác minh tài khoản.
5. **Add Input** → chọn private dataset vừa tạo. Chạy các ô từ trên xuống.

**Mặc định:** đánh giá **toàn bộ validation**, chỉ **Greedy / beam=1** để tiết kiệm quota. Sau đó có thể bật thêm beam 2, 4. Các bản dịch được lưu từng phần tại `/kaggle/working/` (nếu phiên bị ngắt, cần lưu/đính kèm lại output để dùng ở phiên sau).

⚠️ Dữ liệu song ngữ có thông tin được phân phối với điều kiện **CC BY-NC 4.0** và có thể có quyền liên quan từ nguồn gốc. Hãy để dataset và notebook ở chế độ riêng tư, chưa công bố hay dùng thương mại.


## 1. Xác nhận GPU và Kaggle input

In [ ]:
from pathlib import Path
import torch, os
print('Kaggle input:',Path('/kaggle/input').exists())
print('CUDA available:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
    print('VRAM (GiB):',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
else:
    raise RuntimeError('Chưa được cấp Kaggle GPU. Vào Settings → Accelerator để bật GPU.')
print('Một số file input:')
for p in list(Path('/kaggle/input').rglob('*'))[:35]: print(' -',p)

## 2. Cài thư viện tương thích và khắc phục torchao cũ

Bật **Internet: On** trong Kaggle Session options. `HF_TOKEN` không bắt buộc với base model công khai. Nếu pip báo cần restart kernel, hãy restart rồi chạy lại từ mục 1. Không train hoặc xóa file đầu vào.


In [ ]:
%pip -q uninstall -y torchao
%pip -q install "transformers==4.57.6" "peft>=0.17,<1" "accelerate>=1.4,<2" "sacrebleu>=2.5,<3" "pandas>=2,<4" "safetensors>=0.5" tqdm

## 3. Tìm chính xác adapter và validation trong Kaggle Input

In [ ]:
from pathlib import Path
import json, zipfile, shutil, hashlib, re, unicodedata, os, gc
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from IPython.display import display, FileLink

INPUT_ROOT=Path('/kaggle/input')
WORK_DIR=Path('/kaggle/working')
UNZIP_ROOT=WORK_DIR/'vietlangai_unpack'

def find_transfer(root):
    adapters=[]; vals=[]
    for c in root.rglob('adapter_config.json'):
        if any((c.parent/n).is_file() for n in ('adapter_model.safetensors','adapter_model.bin')):
            adapters.append(c.parent)
    vals=list(root.rglob('validation.csv'))
    return adapters, vals

adapter_candidates, val_candidates = find_transfer(INPUT_ROOT)
if not adapter_candidates or not val_candidates:
    zips=list(INPUT_ROOT.rglob('*.zip'))
    for z in zips:
        with zipfile.ZipFile(z) as archive:
            for info in archive.infolist():
                dest=(UNZIP_ROOT/info.filename).resolve()
                assert dest.is_relative_to(UNZIP_ROOT.resolve()), 'Đường dẫn nguy hiểm trong file ZIP'
            archive.extractall(UNZIP_ROOT)
    more_a,more_v = find_transfer(UNZIP_ROOT)
    adapter_candidates += more_a
    val_candidates += more_v

print('Adapter tìm thấy:', *adapter_candidates, sep='\n - ')
print('Validation tìm thấy:', *val_candidates, sep='\n - ')
assert len(adapter_candidates)==1 and len(val_candidates)==1, (
    'Cần có đúng 1 adapter + 1 validation.csv. Hãy kiểm tra các Kaggle Input đã gắn.'
)
ADAPTER_DIR=adapter_candidates[0]
VAL_PATH=val_candidates[0]

def file_sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

# Kiểm tra checksum từ notebook 09A (nếu có)
manifest_candidates=list(INPUT_ROOT.rglob('transfer_manifest.json'))+list(UNZIP_ROOT.rglob('transfer_manifest.json'))
if len(manifest_candidates)==1:
    tmanifest=json.loads(manifest_candidates[0].read_text(encoding='utf-8'))
    for key,path in [('validation.csv',VAL_PATH),('adapter/adapter_config.json',ADAPTER_DIR/'adapter_config.json')]:
        expected=tmanifest.get('files',{}).get(key,{}).get('sha256')
        if expected: assert file_sha256(path)==expected, f'SHA256 không khớp: {key}'
    weight = next(ADAPTER_DIR/n for n in ('adapter_model.safetensors','adapter_model.bin') if (ADAPTER_DIR/n).is_file())
    expected=tmanifest.get('files',{}).get('adapter/'+weight.name,{}).get('sha256')
    if expected: assert file_sha256(weight)==expected, 'Trọng số adapter bị thay đổi sau transfer'
    print('Đã kiểm tra checksum từ transfer_manifest.json')
print('Adapter:',ADAPTER_DIR)
print('Validation:',VAL_PATH)
print('Không sử dụng test.csv')

## 4. Cấu hình đánh giá và dữ liệu Validation

- `MAX_ROWS=None`: toàn bộ validation, dự kiến khoảng **2.295** câu.
- Chạy thử nhỏ bằng `MAX_ROWS=24` trước nếu muốn; file ra nằm ở thư mục khác, không trộn với đánh giá đầy đủ.
- `BEAMS_TO_EVALUATE=(1,)` tiết kiệm GPU. Chạy beam 2/4 sau nếu cần.
- `MAX_NEW_TOKENS=224` để so sánh với notebook Colab 08.


In [ ]:
BASE_MODEL='google/byt5-small'
PREFIX='translate Tay to Vietnamese: '
MAX_SOURCE_LENGTH=256
MAX_NEW_TOKENS=224
MAX_ROWS=None  # Thử đầu tiên có thể đổi thành 24, rồi đổi lại None cho full
BEAMS_TO_EVALUATE=(1,)  # Khi muốn so sánh: (1, 2, 4)
SAVE_EVERY=32
SEED=42

val_all=pd.read_csv(VAL_PATH,dtype=str,keep_default_na=False)
assert {'source_text','target_text'}.issubset(val_all.columns)
val_all=val_all[['source_text','target_text']].reset_index(drop=True)
assert val_all.source_text.str.strip().ne('').all() and val_all.target_text.str.strip().ne('').all()
assert MAX_ROWS is None or 1 <= MAX_ROWS <= len(val_all)
val_df=val_all if MAX_ROWS is None else val_all.iloc[:MAX_ROWS].copy()
EVAL_DIR=WORK_DIR/'vietlangai_eval' / ('full_'+str(len(val_df)) if MAX_ROWS is None else 'pilot_'+str(len(val_df)))
EVAL_DIR.mkdir(parents=True,exist_ok=True)
print('Validation rows:',len(val_df),'/',len(val_all))
print('Output:',EVAL_DIR)
display(val_df.head())

adapter_weights=next(ADAPTER_DIR/n for n in ('adapter_model.safetensors','adapter_model.bin') if (ADAPTER_DIR/n).is_file())
run_manifest={
  'base_model':BASE_MODEL,
  'adapter_sha256':file_sha256(adapter_weights),
  'adapter_config_sha256':file_sha256(ADAPTER_DIR/'adapter_config.json'),
  'validation_sha256':file_sha256(VAL_PATH),
  'sample_count':len(val_df),'prefix':PREFIX,
  'max_source_length':MAX_SOURCE_LENGTH,'max_new_tokens':MAX_NEW_TOKENS,
  'do_sample':False,'length_penalty':1.0,'seed':SEED,
  'test_used':False,
}
manifest_file=EVAL_DIR/'eval_manifest.json'
if manifest_file.exists():
    prev=json.loads(manifest_file.read_text(encoding='utf-8'))
    assert prev==run_manifest,'Lịch sử cũ khác cấu hình: chọn EVAL_DIR mới'
else:
    manifest_file.write_text(json.dumps(run_manifest,ensure_ascii=False,indent=2),encoding='utf-8')

## 5. Tải ByT5 gốc và LoRA adapter để suy luận (không train)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

gc.collect()
torch.cuda.empty_cache()
try:
    tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL)
    base_model=AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL,use_safetensors=True)
except Exception as exc:
    raise RuntimeError('Không tải được ByT5 từ Hugging Face. Kiểm tra Kaggle Internet=On. Nếu bị giới hạn mạng, cần tải model gốc thành một Kaggle Dataset riêng.') from exc
model=PeftModel.from_pretrained(base_model,str(ADAPTER_DIR),is_trainable=False)
assert isinstance(model,PeftModel)
assert not any(p.requires_grad for p in model.parameters())
model=model.to('cuda').eval()
device=next(model.parameters()).device
print('Loaded:',type(model).__name__)
print('Device:',device)
print('Base model:',BASE_MODEL)
print('Adapter:',ADAPTER_DIR)

## 6. Sinh bản dịch và lưu từng 32 câu (tránh mất kết quả khi ngắt phiên)

In [ ]:
CACHE_COLUMNS=['row_id','source_text','target_text','model_prediction']

def read_cache(path):
    if not path.exists():return pd.DataFrame(columns=CACHE_COLUMNS)
    df=pd.read_csv(path,keep_default_na=False,dtype={'row_id':int})
    assert list(df.columns)==CACHE_COLUMNS
    assert len(df)<=len(val_df)
    assert df.row_id.tolist()==list(range(len(df))), 'Cache row_id không liên tiếp'
    assert df.source_text.tolist()==val_df.source_text.iloc[:len(df)].tolist()
    assert df.target_text.tolist()==val_df.target_text.iloc[:len(df)].tolist()
    return df

def atomic_csv(frame,path):
    temp=path.with_suffix('.tmp')
    frame.to_csv(temp,index=False,encoding='utf-8-sig')
    os.replace(temp,path)

def evaluate_beam(beam):
    assert beam in (1,2,4)
    out=EVAL_DIR/f'beam_{beam}_predictions.csv'
    cached=read_cache(out)
    rows=cached.to_dict('records')
    start=len(rows)
    batch_size={1:4,2:2,4:1}[beam]
    print(f'Beam={beam}: bắt đầu từ {start}/{len(val_df)} câu')
    for i in tqdm(range(start,len(val_df),batch_size),desc=f'beam {beam}'):
        chunk=val_df.iloc[i:i+batch_size]
        encoded=tokenizer(
            [PREFIX+s for s in chunk.source_text.tolist()],
            padding=True,truncation=True,max_length=MAX_SOURCE_LENGTH,
            return_tensors='pt'
        ).to(device)
        opts={'num_beams':beam,'do_sample':False,'max_new_tokens':MAX_NEW_TOKENS,'length_penalty':1.0}
        if beam>1:opts['early_stopping']=True
        with torch.inference_mode():
            pred_ids=model.generate(**encoded,**opts)
        pred=tokenizer.batch_decode(pred_ids,skip_special_tokens=True)
        for offset,p in enumerate(pred):
            j=i+offset
            rows.append({'row_id':j,'source_text':val_df.at[j,'source_text'],
                         'target_text':val_df.at[j,'target_text'],'model_prediction':p})
        if len(rows)%SAVE_EVERY<batch_size or len(rows)==len(val_df):
            atomic_csv(pd.DataFrame(rows,columns=CACHE_COLUMNS),out)
    assert len(read_cache(out))==len(val_df),f'Beam={beam} chưa hoàn thành'
    print('Đã lưu:',out)

for beam in BEAMS_TO_EVALUATE:
    evaluate_beam(beam)

## 7. Tính chrF++, BLEU, Exact Match, lặp từ và xuất báo cáo

In [ ]:
from sacrebleu.metrics import CHRF, BLEU
chrf=CHRF(word_order=2)
bleu=BLEU(effective_order=True)

def norm(s):return re.sub(r'\s+',' ',unicodedata.normalize('NFC',str(s))).strip().casefold()
def repeated3(s):
    w=str(s).casefold().split()
    return any(w[i]==w[i+1]==w[i+2] for i in range(len(w)-2))

def scores(preds,refs):
    return {
        'n':len(preds),
        'chrf_plus_plus':float(chrf.corpus_score(preds,[refs]).score),
        'bleu':float(bleu.corpus_score(preds,[refs]).score),
        'exact_match_rate':float(np.mean([norm(p)==norm(r) for p,r in zip(preds,refs)])),
        'repeat_3_rate':float(np.mean([repeated3(p) for p in preds])),
        'empty_rate':float(np.mean([not norm(p) for p in preds]))
    }

score_rows=[]
combined=val_df.copy()
combined.insert(0,'row_id',range(len(combined)))
for beam in (1,2,4):
    csv=EVAL_DIR/f'beam_{beam}_predictions.csv'
    if not csv.exists():continue
    cache=read_cache(csv)
    if len(cache)<len(val_df):
        print(f'Chưa tính Beam={beam}: {len(cache)}/{len(val_df)} câu')
        continue
    preds=cache.model_prediction.tolist()
    s=scores(preds,val_df.target_text.tolist())
    score_rows.append({'beam':beam,**s})
    combined[f'prediction_beam_{beam}']=preds

assert score_rows,'Chưa có beam hoàn tất; hãy chạy bước 6'
score_df=pd.DataFrame(score_rows)
score_df.to_csv(EVAL_DIR/'scores_full_validation.csv',index=False,encoding='utf-8-sig')
combined['source_words']=combined.source_text.str.split().str.len()
combined['length_group']=pd.cut(combined.source_words,bins=[0,1,4,float('inf')],labels=['1_word','2_to_4_words','5_plus_words'])
combined.to_csv(EVAL_DIR/'full_validation_comparison.csv',index=False,encoding='utf-8-sig')
length_results=[]
for group,sub in combined.groupby('length_group',observed=True):
    for beam in score_df.beam.tolist():
        pred_col=f'prediction_beam_{beam}'
        length_results.append({'length_group':str(group),'beam':int(beam),**scores(sub[pred_col].tolist(),sub.target_text.tolist())})
length_df=pd.DataFrame(length_results)
length_df.to_csv(EVAL_DIR/'metrics_by_length.csv',index=False,encoding='utf-8-sig')
review=combined.groupby('length_group',observed=True,group_keys=False).apply(lambda x:x.sample(min(20,len(x)),random_state=42))
review.to_csv(EVAL_DIR/'review_samples.csv',index=False,encoding='utf-8-sig')
print('====== SCORES ======')
display(score_df)
print('====== 15 MẪU DỊCH ======')
display(review.head(15))
print('Kết quả tại:',EVAL_DIR)

## 8. Nén và tải kết quả về máy hoặc lưu Output của Kaggle

Nhấn **Save Version** (chọn lưu output) hoặc tải ZIP trong ô dưới về máy. `/kaggle/working/` không phải chỗ lưu bền vững giữa mọi phiên; nếu muốn tiếp tục ở phiên mới, hãy gắn output cũ như một Kaggle Input và sao chép file cache tương ứng sang EVAL_DIR trước khi chạy bước 6.

In [ ]:
import shutil
result_zip=WORK_DIR/'VietLangAI_Kaggle_Validation_Results.zip'
shutil.make_archive(str(result_zip.with_suffix('')),'zip',root_dir=WORK_DIR/'vietlangai_eval')
print('Tải từ Output:',result_zip)
display(FileLink(str(result_zip)))

In [ ]:
from pathlib import Path
import torch

print("GPU available:", torch.cuda.is_available())

if "model" in globals():
    print("Model device:", next(model.parameters()).device)

print("\n=== CÁC FILE KẾT QUẢ ===")

root = Path("/kaggle/working")

for p in root.rglob("*"):
    if p.is_file():
        print(p.relative_to(root), 
              f"({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
from pathlib import Path
from difflib import SequenceMatcher
import pandas as pd

required = {
    "source_text",
    "target_text",
    "prediction_beam_1"
}

candidates = []

for path in Path("/kaggle/working").rglob("*.csv"):
    try:
        data = pd.read_csv(path, keep_default_na=False)
        if required.issubset(data.columns):
            candidates.append((len(data), path, data))
    except Exception:
        pass

assert candidates, "Không tìm thấy file dự đoán."

# Ưu tiên bảng đầy đủ, không dùng bảng 60 câu mẫu
candidates.sort(key=lambda x: x[0], reverse=True)
n, path, df = candidates[0]

print("File:", path)
print("Số câu:", n)

def similarity(a, b):
    return SequenceMatcher(
        None,
        str(a).lower().strip(),
        str(b).lower().strip()
    ).ratio()

df["source_words"] = df["source_text"].str.split().str.len()

df["copy_similarity"] = [
    similarity(a, b)
    for a, b in zip(
        df["source_text"],
        df["prediction_beam_1"]
    )
]

# Chỉ đánh giá sao chép với câu từ 2 từ trở lên
longer = df[df["source_words"] >= 2].copy()

longer["suspected_copy"] = (
    longer["copy_similarity"] >= 0.7
)

print("\n=== SOURCE COPYING ===")
print(
    "Tỷ lệ nghi sao chép:",
    f"{longer['suspected_copy'].mean():.2%}"
)

print("\n=== CÁC DỰ ĐOÁN PHỔ BIẾN ===")
print(df["prediction_beam_1"].value_counts().head(15))

print("\n=== MẪU NGHI SAO CHÉP ===")
display(
    longer[longer["suspected_copy"]][[
        "source_text",
        "target_text",
        "prediction_beam_1",
        "copy_similarity"
    ]].head(20)
)

In [ ]:
from pathlib import Path
import pandas as pd

# Tìm file kết quả dịch đã xuất
candidates = []

for p in Path("/kaggle/working").rglob("*.csv"):
    try:
        cols = set(pd.read_csv(p, nrows=1).columns)
        required = {
            "source_text",
            "target_text",
            "prediction_beam_1"
        }

        if required.issubset(cols):
            candidates.append(p)
    except Exception:
        pass

assert candidates, "Chưa tìm thấy CSV. Hãy chạy bước xuất kết quả."

path = max(candidates, key=lambda p: p.stat().st_mtime)
df = pd.read_csv(path).fillna("")

df["source_words"] = (
    df["source_text"].astype(str).str.split().str.len()
)

groups = {
    "TỪ ĐƠN": df["source_words"] == 1,
    "CỤM NGẮN (2–4 TỪ)": df["source_words"].between(2, 4),
    "CÂU DÀI (>=5 TỪ)": df["source_words"] >= 5,
}

for name, mask in groups.items():
    subset = df.loc[mask]

    print(f"\n===== {name}: {len(subset)} CÂU =====")

    display(
        subset[[
            "source_text",
            "target_text",
            "prediction_beam_1"
        ]].sample(
            n=min(10, len(subset)),
            random_state=42
        ) if len(subset) else subset
    )

print("\n===== 10 BẢN DỊCH XUẤT HIỆN NHIỀU NHẤT =====")
print(df["prediction_beam_1"].value_counts().head(10))

In [ ]:
from pathlib import Path

print("=== KIỂM TRA MODEL TRÊN KAGGLE ===")

for root in ["/kaggle/input", "/kaggle/working"]:
    for p in Path(root).rglob("*"):
        if p.is_file() and (
            p.name in [
                "adapter_config.json",
                "adapter_model.safetensors"
            ]
            or p.name.endswith(".zip")
        ):
            print(p)